# HAR FPGA All-in-One Notebook

This notebook is self-contained. Run all cells to download the dataset, train all 5 models, save artifacts, export FPGA weight files, run post-training quantization, and generate comparison plots.

Outputs are written to `./artifacts/` relative to the notebook working directory.

In [ ]:
%pip install -q numpy requests matplotlib scikit-learn pywavelets plotly tensorflow

In [ ]:
from __future__ import annotations

import json
import random
import struct
import time
import zipfile
from pathlib import Path

import numpy as np
import requests
import tensorflow as tf
import pywt
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.offline import plot as plotly_save

from tensorflow import keras
from tensorflow.keras import layers

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
ARTIFACTS_ROOT = ROOT / 'artifacts'
ZIP_PATH = DATA_DIR / 'UCI_HAR_Dataset.zip'
EXTRACT_DIR = DATA_DIR / 'UCI HAR Dataset'
DATASET_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip'
MODEL_TYPES = ('mlp', '1dcnn', '2dcnn', 'cnn_lstm', 'wclstm')
SIGNAL_FILES = [
    'body_acc_x', 'body_acc_y', 'body_acc_z',
    'body_gyro_x', 'body_gyro_y', 'body_gyro_z',
    'total_acc_x', 'total_acc_y', 'total_acc_z',
]

FEATURE_CONFIG = {
    'feature_indices_1based': [1, 2, 3, 4, 5, 6, 41, 42, 43, 44, 45, 46, 201, 202, 214, 215, 227, 228, 253],
    'feature_names': [
        'tBodyAcc-mean()-X', 'tBodyAcc-mean()-Y', 'tBodyAcc-mean()-Z',
        'tBodyAcc-std()-X', 'tBodyAcc-std()-Y', 'tBodyAcc-std()-Z',
        'tGravityAcc-mean()-X', 'tGravityAcc-mean()-Y', 'tGravityAcc-mean()-Z',
        'tGravityAcc-std()-X', 'tGravityAcc-std()-Y', 'tGravityAcc-std()-Z',
        'tBodyAccMag-mean()', 'tBodyAccMag-std()',
        'tGravityAccMag-mean()', 'tGravityAccMag-std()',
        'tBodyAccJerkMag-mean()', 'tBodyAccJerkMag-std()', 'tBodyGyroMag-mean()'
    ],
}

TRAINING_CONFIG = {
    'epochs': 30,
    'batch_size': 64,
    'learning_rate': 0.001,
    'validation_split': 0.2,
    'random_seed': 42,
    'loss': 'sparse_categorical_crossentropy',
    'class_names': ['WALKING', 'SITTING', 'STANDING', 'LAYING', 'TRANSITION'],
    'label_map': {'1': 0, '2': 0, '3': 0, '4': 1, '5': 2, '6': 3},
    'models': {
        'mlp': {
            'data_mode': 'features', 'input_length': 19, 'hidden1_units': 64, 'hidden2_units': 32, 'hidden3_units': 16,
            'hidden_activation': 'relu', 'dropout_rate': 0.3, 'num_classes': 5, 'output_activation': 'softmax'
        },
        '1dcnn': {
            'data_mode': 'features', 'input_length': 19, 'conv1_filters': 12, 'conv1_kernel_size': 19, 'conv1_stride': 1,
            'conv1_padding': 'valid', 'conv2_filters': 8, 'conv2_kernel_size': 1, 'conv2_stride': 1, 'conv2_padding': 'valid',
            'conv_activation': 'relu', 'num_classes': 5, 'output_activation': 'softmax'
        },
        '2dcnn': {
            'data_mode': 'raw', 'input_timesteps': 128, 'input_channels': 9, 'conv1_filters': 16, 'conv1_kernel': (3, 3),
            'conv2_filters': 32, 'conv2_kernel': (3, 3), 'conv_activation': 'relu', 'dropout_rate': 0.3,
            'num_classes': 5, 'output_activation': 'softmax'
        },
        'cnn_lstm': {
            'data_mode': 'raw', 'input_timesteps': 128, 'input_channels': 9, 'conv1_filters': 64, 'conv1_kernel_size': 5,
            'conv2_filters': 64, 'conv2_kernel_size': 5, 'conv_activation': 'relu', 'lstm_units': 128, 'dropout_rate': 0.3,
            'num_classes': 5, 'output_activation': 'softmax'
        },
        'wclstm': {
            'data_mode': 'wavelet', 'wavelet': 'db4', 'wavelet_level': 2, 'input_channels': 9, 'conv1_filters': 64,
            'conv1_kernel_size': 5, 'conv2_filters': 64, 'conv2_kernel_size': 5, 'conv_activation': 'relu',
            'lstm_units': 128, 'dropout_rate': 0.3, 'num_classes': 5, 'output_activation': 'softmax'
        },
    },
}

TIMING_RUNS = 10
WARMUP_RUNS = 3


def artifact_dir(model_type: str) -> Path:
    return ARTIFACTS_ROOT / model_type


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def maybe_enable_gpu() -> None:
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        print('GPUs:', [g.name for g in gpus])
        for gpu in gpus:
            try:
                tf.config.experimental.set_memory_growth(gpu, True)
            except Exception:
                pass
    else:
        print('No GPU detected. Using CPU.')


def download_dataset(force: bool = False) -> Path:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    if ZIP_PATH.exists() and not force:
        print(f'[data] Zip already cached at {ZIP_PATH}')
        return ZIP_PATH
    print(f'[data] Downloading dataset from {DATASET_URL} ...')
    resp = requests.get(DATASET_URL, stream=True, timeout=120)
    resp.raise_for_status()
    with open(ZIP_PATH, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=1 << 16):
            f.write(chunk)
    print(f'[data] Saved to {ZIP_PATH}')
    return ZIP_PATH


def extract_dataset(force: bool = False) -> Path:
    if EXTRACT_DIR.exists() and not force:
        print(f'[data] Already extracted at {EXTRACT_DIR}')
        return EXTRACT_DIR
    print('[data] Extracting zip ...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(DATA_DIR)
    print(f'[data] Extracted to {EXTRACT_DIR}')
    return EXTRACT_DIR


def remap_labels(y: np.ndarray) -> np.ndarray:
    y_new = np.full_like(y, fill_value=4)
    for orig_str, new_id in TRAINING_CONFIG['label_map'].items():
        y_new[y == int(orig_str)] = new_id
    return y_new


def load_har_data(download: bool = True):
    if download:
        download_dataset()
        extract_dataset()
    train_dir = EXTRACT_DIR / 'train'
    test_dir = EXTRACT_DIR / 'test'
    X_train_full = np.loadtxt(train_dir / 'X_train.txt')
    X_test_full = np.loadtxt(test_dir / 'X_test.txt')
    y_train_raw = np.loadtxt(train_dir / 'y_train.txt', dtype=int)
    y_test_raw = np.loadtxt(test_dir / 'y_test.txt', dtype=int)
    feat_idx = [i - 1 for i in FEATURE_CONFIG['feature_indices_1based']]
    X_train = X_train_full[:, feat_idx].astype(np.float32)
    X_test = X_test_full[:, feat_idx].astype(np.float32)
    y_train = remap_labels(y_train_raw)
    y_test = remap_labels(y_test_raw)
    return X_train, y_train, X_test, y_test, FEATURE_CONFIG['feature_names'], TRAINING_CONFIG['class_names']


def load_har_raw(download: bool = True):
    if download:
        download_dataset()
        extract_dataset()
    train_dir = EXTRACT_DIR / 'train'
    test_dir = EXTRACT_DIR / 'test'
    X_train = np.stack([np.loadtxt(train_dir / 'Inertial Signals' / f'{name}_train.txt') for name in SIGNAL_FILES], axis=-1).astype(np.float32)
    X_test = np.stack([np.loadtxt(test_dir / 'Inertial Signals' / f'{name}_test.txt') for name in SIGNAL_FILES], axis=-1).astype(np.float32)
    y_train = remap_labels(np.loadtxt(train_dir / 'y_train.txt', dtype=int))
    y_test = remap_labels(np.loadtxt(test_dir / 'y_test.txt', dtype=int))
    return X_train, y_train, X_test, y_test, SIGNAL_FILES, TRAINING_CONFIG['class_names']


def apply_wavelet_transform(X: np.ndarray, wavelet: str = 'db4', level: int = 2) -> np.ndarray:
    n, _, c = X.shape
    sample = np.concatenate(pywt.wavedec(X[0, :, 0], wavelet, level=level))
    out = np.empty((n, len(sample), c), dtype=np.float32)
    for i in range(n):
        for ch in range(c):
            out[i, :, ch] = np.concatenate(pywt.wavedec(X[i, :, ch], wavelet, level=level))
    return out


class ZScoreScaler:
    def __init__(self):
        self.mean_ = None
        self.std_ = None

    def fit(self, X: np.ndarray):
        self.mean_ = X.mean(axis=0).astype(np.float64)
        self.std_ = X.std(axis=0).astype(np.float64)
        self.std_[self.std_ == 0.0] = 1.0
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        return ((X - self.mean_) / self.std_).astype(np.float32)

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        return self.fit(X).transform(X)

    def save(self, path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, 'w') as f:
            json.dump({'mean': self.mean_.tolist(), 'std': self.std_.tolist()}, f, indent=2)

    @classmethod
    def load(cls, path: Path):
        with open(path, 'r') as f:
            data = json.load(f)
        scaler = cls()
        scaler.mean_ = np.array(data['mean'], dtype=np.float64)
        scaler.std_ = np.array(data['std'], dtype=np.float64)
        return scaler


def build_mlp(input_length=19, hidden1_units=64, hidden2_units=32, hidden3_units=16, hidden_activation='relu', dropout_rate=0.3, num_classes=5, output_activation='softmax'):
    inp = keras.Input(shape=(input_length,), name='input')
    x = layers.Dense(hidden1_units, activation=hidden_activation, name='dense_1')(inp)
    x = layers.Dropout(dropout_rate, name='dropout_1')(x)
    x = layers.Dense(hidden2_units, activation=hidden_activation, name='dense_2')(x)
    x = layers.Dropout(dropout_rate, name='dropout_2')(x)
    x = layers.Dense(hidden3_units, activation=hidden_activation, name='dense_3')(x)
    x = layers.Dropout(dropout_rate, name='dropout_3')(x)
    out = layers.Dense(num_classes, activation=output_activation, name='dense_output')(x)
    return keras.Model(inp, out, name='har_mlp')


def build_1dcnn(input_length=19, conv1_filters=12, conv1_kernel_size=19, conv1_stride=1, conv1_padding='valid', conv2_filters=8, conv2_kernel_size=1, conv2_stride=1, conv2_padding='valid', conv_activation='relu', num_classes=5, output_activation='softmax'):
    inp = keras.Input(shape=(input_length, 1), name='input')
    x = layers.Conv1D(conv1_filters, conv1_kernel_size, strides=conv1_stride, padding=conv1_padding, activation=conv_activation, name='conv1d_1')(inp)
    x = layers.Conv1D(conv2_filters, conv2_kernel_size, strides=conv2_stride, padding=conv2_padding, activation=conv_activation, name='conv1d_2')(x)
    x = layers.Flatten(name='flatten')(x)
    out = layers.Dense(num_classes, activation=output_activation, name='dense_output')(x)
    return keras.Model(inp, out, name='har_1dcnn')


def build_2dcnn(input_timesteps=128, input_channels=9, conv1_filters=16, conv1_kernel=(3, 3), conv2_filters=32, conv2_kernel=(3, 3), conv_activation='relu', dropout_rate=0.3, num_classes=5, output_activation='softmax'):
    inp = keras.Input(shape=(input_timesteps, input_channels, 1), name='input')
    x = layers.Conv2D(conv1_filters, conv1_kernel, padding='same', activation=conv_activation, name='conv2d_1')(inp)
    x = layers.MaxPooling2D(pool_size=(2, 1), name='maxpool_1')(x)
    x = layers.Conv2D(conv2_filters, conv2_kernel, padding='same', activation=conv_activation, name='conv2d_2')(x)
    x = layers.MaxPooling2D(pool_size=(2, 1), name='maxpool_2')(x)
    x = layers.Dropout(dropout_rate, name='dropout_1')(x)
    x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    out = layers.Dense(num_classes, activation=output_activation, name='dense_output')(x)
    return keras.Model(inp, out, name='har_2dcnn')


def build_cnn_lstm(input_timesteps=128, input_channels=9, conv1_filters=64, conv1_kernel_size=5, conv2_filters=64, conv2_kernel_size=5, conv_activation='relu', lstm_units=128, dropout_rate=0.3, num_classes=5, output_activation='softmax'):
    inp = keras.Input(shape=(input_timesteps, input_channels), name='input')
    x = layers.Conv1D(conv1_filters, conv1_kernel_size, padding='same', activation=conv_activation, name='conv1d_1')(inp)
    x = layers.Conv1D(conv2_filters, conv2_kernel_size, padding='same', activation=conv_activation, name='conv1d_2')(x)
    x = layers.Dropout(dropout_rate, name='dropout_1')(x)
    x = layers.LSTM(lstm_units, name='lstm')(x)
    x = layers.Dropout(dropout_rate, name='dropout_2')(x)
    out = layers.Dense(num_classes, activation=output_activation, name='dense_output')(x)
    return keras.Model(inp, out, name='har_cnn_lstm')


def build_wclstm(input_timesteps=141, input_channels=9, conv1_filters=64, conv1_kernel_size=5, conv2_filters=64, conv2_kernel_size=5, conv_activation='relu', lstm_units=128, dropout_rate=0.3, num_classes=5, output_activation='softmax'):
    return build_cnn_lstm(input_timesteps=input_timesteps, input_channels=input_channels, conv1_filters=conv1_filters, conv1_kernel_size=conv1_kernel_size, conv2_filters=conv2_filters, conv2_kernel_size=conv2_kernel_size, conv_activation=conv_activation, lstm_units=lstm_units, dropout_rate=dropout_rate, num_classes=num_classes, output_activation=output_activation)


def build_model(model_type: str, **kwargs):
    if model_type == 'mlp':
        return build_mlp(**kwargs)
    if model_type == '1dcnn':
        return build_1dcnn(**kwargs)
    if model_type == '2dcnn':
        return build_2dcnn(**kwargs)
    if model_type == 'cnn_lstm':
        return build_cnn_lstm(**kwargs)
    if model_type == 'wclstm':
        return build_wclstm(**kwargs)
    raise ValueError(model_type)


def extract_model_spec(model: keras.Model) -> dict:
    spec = {
        'model_name': model.name,
        'total_params': int(model.count_params()),
        'input_shape': list(model.input_shape),
        'output_shape': list(model.output_shape),
        'layers': [],
    }
    for layer in model.layers:
        cfg = layer.get_config()
        weight_shapes = {w.name: list(w.shape) for w in layer.weights}
        try:
            out_shape = list(layer.output_shape)
        except Exception:
            out_shape = list(layer.output.shape) if hasattr(layer, 'output') else []
        entry = {
            'name': layer.name,
            'type': layer.__class__.__name__,
            'output_shape': out_shape,
            'param_count': int(layer.count_params()),
            'weight_shapes': weight_shapes,
            'config': {},
        }
        if isinstance(layer, (layers.Conv1D, layers.Conv2D)):
            entry['config'] = {'filters': cfg['filters'], 'kernel_size': cfg['kernel_size'], 'strides': cfg['strides'], 'padding': cfg['padding'], 'activation': cfg['activation'], 'use_bias': cfg['use_bias']}
        elif isinstance(layer, layers.Dense):
            entry['config'] = {'units': cfg['units'], 'activation': cfg['activation'], 'use_bias': cfg['use_bias']}
        elif isinstance(layer, layers.LSTM):
            entry['config'] = {'units': cfg['units'], 'activation': cfg['activation'], 'recurrent_activation': cfg['recurrent_activation'], 'use_bias': cfg['use_bias'], 'return_sequences': cfg['return_sequences']}
        elif isinstance(layer, (layers.Dropout,)):
            entry['config'] = {'rate': cfg['rate']}
        elif isinstance(layer, (layers.MaxPooling1D, layers.MaxPooling2D)):
            entry['config'] = {'pool_size': cfg['pool_size'], 'strides': cfg['strides'], 'padding': cfg['padding']}
        spec['layers'].append(entry)
    return spec


def save_training_curves(history, out_path: Path, model_type: str) -> None:
    hist = history.history
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(hist['loss'], label='train')
    axes[0].plot(hist['val_loss'], label='val')
    axes[0].set_title(f'{model_type.upper()} Loss')
    axes[0].legend()
    axes[1].plot(hist['accuracy'], label='train')
    axes[1].plot(hist['val_accuracy'], label='val')
    axes[1].set_title(f'{model_type.upper()} Accuracy')
    axes[1].legend()
    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)


def preprocess_for_model(model_type: str, X_train: np.ndarray, X_test: np.ndarray):
    scaler = ZScoreScaler()
    if X_train.ndim == 3:
        n_tr, t, c = X_train.shape
        n_te = X_test.shape[0]
        X_train_2d = X_train.reshape(n_tr, t * c)
        X_test_2d = X_test.reshape(n_te, t * c)
        X_train = scaler.fit_transform(X_train_2d).reshape(n_tr, t, c)
        X_test = scaler.transform(X_test_2d).reshape(n_te, t, c)
    else:
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
    if model_type == '1dcnn':
        X_train = X_train[..., np.newaxis]
        X_test = X_test[..., np.newaxis]
    if model_type == '2dcnn':
        X_train = X_train[..., np.newaxis]
        X_test = X_test[..., np.newaxis]
    return scaler, X_train, X_test


def train_one_model(model_type: str, epochs: int | None = None, batch_size: int | None = None, learning_rate: float | None = None):
    cfg = TRAINING_CONFIG
    model_cfg = cfg['models'][model_type].copy()
    data_mode = model_cfg['data_mode']
    epochs = epochs or cfg['epochs']
    batch_size = batch_size or cfg['batch_size']
    learning_rate = learning_rate or cfg['learning_rate']
    seed_everything(cfg['random_seed'])

    if data_mode == 'features':
        X_train, y_train, X_test, y_test, feature_names, class_names = load_har_data()
    else:
        X_train, y_train, X_test, y_test, feature_names, class_names = load_har_raw()

    if data_mode == 'wavelet':
        X_train = apply_wavelet_transform(X_train, wavelet=model_cfg.get('wavelet', 'db4'), level=model_cfg.get('wavelet_level', 2))
        X_test = apply_wavelet_transform(X_test, wavelet=model_cfg.get('wavelet', 'db4'), level=model_cfg.get('wavelet_level', 2))

    scaler, X_train, X_test = preprocess_for_model(model_type, X_train, X_test)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=cfg['validation_split'], random_state=cfg['random_seed'], stratify=y_train
    )

    build_kwargs = {k: v for k, v in model_cfg.items() if k not in ('data_mode', 'wavelet', 'wavelet_level')}
    if model_type == 'wclstm':
        build_kwargs['input_timesteps'] = X_train.shape[1]

    model = build_model(model_type, **build_kwargs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss=cfg['loss'], metrics=['accuracy'])

    t0 = time.time()
    history = model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=epochs, batch_size=batch_size, verbose=2)
    elapsed = time.time() - t0

    train_loss, train_acc = model.evaluate(X_tr, y_tr, verbose=0)
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

    adir = artifact_dir(model_type)
    adir.mkdir(parents=True, exist_ok=True)
    scaler.save(adir / 'scaler.json')
    model.save(adir / 'har_model.keras')

    history_dict = {k: [float(v) for v in vals] for k, vals in history.history.items()}
    with open(adir / 'training_history.json', 'w') as f:
        json.dump(history_dict, f, indent=2)

    spec = extract_model_spec(model)
    spec['model_type'] = model_type
    spec['training_info'] = {
        'epochs': epochs, 'batch_size': batch_size, 'learning_rate': learning_rate,
        'final_train_accuracy': float(train_acc), 'final_test_accuracy': float(test_acc), 'final_test_loss': float(test_loss),
        'training_time_seconds': round(elapsed, 2), 'data_mode': data_mode, 'feature_names': feature_names,
        'class_names': class_names, 'num_classes': len(class_names),
    }
    if data_mode == 'wavelet':
        spec['training_info']['wavelet'] = model_cfg.get('wavelet', 'db4')
        spec['training_info']['wavelet_level'] = model_cfg.get('wavelet_level', 2)
        spec['training_info']['wavelet_output_timesteps'] = int(X_train.shape[1])
    with open(adir / 'model_spec.json', 'w') as f:
        json.dump(spec, f, indent=2)

    save_training_curves(history, adir / 'training_curves.png', model_type)

    print(f'[train] {model_type}: test_acc={test_acc:.4f}, params={model.count_params()}, time={elapsed:.1f}s')
    return {'model_type': model_type, 'model': model, 'X_test': X_test, 'y_test': y_test, 'class_names': class_names, 'scaler': scaler, 'spec': spec}


def float32_to_hex(value: float) -> str:
    return format(struct.unpack('>I', struct.pack('>f', value))[0], '08x')


def float16_to_hex(value) -> str:
    return format(struct.unpack('>H', struct.pack('>e', np.float16(value)))[0], '04x')


def int16_to_hex(value: int) -> str:
    return format(value & 0xFFFF, '04x')


def int8_to_hex(value: int) -> str:
    return format(value & 0xFF, '02x')


def export_spec_json(model: keras.Model, path: Path) -> None:
    training_info = None
    if path.exists():
        with open(path, 'r') as f:
            training_info = json.load(f).get('training_info')
    spec = extract_model_spec(model)
    if training_info is not None:
        spec['training_info'] = training_info
    with open(path, 'w') as f:
        json.dump(spec, f, indent=2)


def export_weights_mem(model: keras.Model, path: Path) -> None:
    lines = [f'// {model.name} Model Weights', f'// Total parameters: {model.count_params()}', '']
    for layer in model.layers:
        for w in layer.weights:
            arr = w.numpy().astype(np.float32).flatten()
            shape_str = 'x'.join(str(d) for d in w.shape)
            lines.append(f'// {w.name}  shape=({shape_str})  count={arr.size}')
            lines.extend(float32_to_hex(float(v)) for v in arr)
            lines.append('')
    with open(path, 'w') as f:
        f.write('\n'.join(lines))


def export_weights_readable(model: keras.Model, path: Path) -> None:
    lines = [f'{model.name} Model Weights', f'Total parameters: {model.count_params()}', '=' * 60]
    for layer in model.layers:
        if not layer.weights:
            continue
        lines.append(f'\nLayer: {layer.name} ({layer.__class__.__name__})')
        for w in layer.weights:
            arr = w.numpy()
            lines.append(f'  {w.name} shape={list(w.shape)}')
            if arr.size <= 100:
                flat = arr.flatten()
                for i in range(0, len(flat), 8):
                    chunk = flat[i:i + 8]
                    lines.append('    ' + '  '.join(f'{v:+.6f}' for v in chunk))
            else:
                lines.append(f'    min={arr.min():.6f} max={arr.max():.6f} mean={arr.mean():.6f} std={arr.std():.6f}')
    with open(path, 'w') as f:
        f.write('\n'.join(lines))


def export_model_artifacts(model_type: str, model: keras.Model | None = None) -> None:
    adir = artifact_dir(model_type)
    adir.mkdir(parents=True, exist_ok=True)
    if model is None:
        model = keras.models.load_model(adir / 'har_model.keras')
    export_spec_json(model, adir / 'model_spec.json')
    export_weights_mem(model, adir / 'model_weights.mem')
    export_weights_readable(model, adir / 'weights_readable.txt')


def quantize_symmetric(arr: np.ndarray, n_bits: int):
    qmin = -(2 ** (n_bits - 1)) + 1
    qmax = 2 ** (n_bits - 1) - 1
    abs_max = max(abs(arr.min()), abs(arr.max()))
    if abs_max == 0:
        return np.zeros_like(arr, dtype=np.int32), 1.0
    scale = abs_max / qmax
    quantized = np.clip(np.round(arr / scale), qmin, qmax).astype(np.int32)
    return quantized, float(scale)


def dequantize(q: np.ndarray, scale: float) -> np.ndarray:
    return q.astype(np.float32) * scale


def extract_weights(model: keras.Model) -> dict[str, np.ndarray]:
    result = {}
    for layer in model.layers:
        for w in layer.weights:
            result[f'{layer.name}/{w.name}'] = w.numpy().astype(np.float32)
    return result


def set_weights_from_dict(model: keras.Model, wd: dict[str, np.ndarray]) -> None:
    for layer in model.layers:
        weights = []
        for w in layer.weights:
            weights.append(wd[f'{layer.name}/{w.name}'].astype(np.float32))
        if weights:
            layer.set_weights(weights)


def build_quantized_weights(original: dict[str, np.ndarray], variant: str) -> dict[str, np.ndarray]:
    result = {}
    for name, arr in original.items():
        if variant == 'fp32':
            result[name] = arr.copy()
        elif variant == 'fp16':
            result[name] = arr.astype(np.float16).astype(np.float32)
        elif variant == 'int16':
            q, s = quantize_symmetric(arr, 16)
            result[name] = dequantize(q, s)
        elif variant == 'int8':
            q, s = quantize_symmetric(arr, 8)
            result[name] = dequantize(q, s)
        else:
            raise ValueError(variant)
    return result


def timed_inference(model: keras.Model, X: np.ndarray):
    for _ in range(WARMUP_RUNS):
        model.predict(X, verbose=0)
    times = []
    preds = None
    for _ in range(TIMING_RUNS):
        t0 = time.perf_counter()
        preds = model.predict(X, verbose=0)
        times.append(time.perf_counter() - t0)
    return preds, float(np.mean(times))


def weight_size_bytes(weights_dict: dict[str, np.ndarray], variant: str) -> int:
    bits = {'fp32': 32, 'fp16': 16, 'int16': 16, 'int8': 8}[variant]
    return sum(arr.size for arr in weights_dict.values()) * bits // 8


def write_mem_fp16(weights_dict: dict[str, np.ndarray], out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    lines = ['// HAR Model FP16 quantized weights', '']
    metadata = {'dtype': 'float16', 'layers': {}}
    for name, arr in weights_dict.items():
        flat = arr.astype(np.float16).flatten()
        shape_str = 'x'.join(str(d) for d in arr.shape)
        lines.append(f'// {name} shape=({shape_str}) count={flat.size}')
        lines.extend(float16_to_hex(v) for v in flat)
        lines.append('')
        metadata['layers'][name] = {'shape': list(arr.shape), 'count': int(flat.size)}
    with open(out_dir / 'model_weights_fp16.mem', 'w') as f:
        f.write('\n'.join(lines))
    with open(out_dir / 'metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)


def write_mem_int(weights_dict: dict[str, np.ndarray], n_bits: int, out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    hex_fn = int8_to_hex if n_bits == 8 else int16_to_hex
    metadata = {'dtype': f'int{n_bits}', 'layers': {}}
    lines = [f'// HAR Model INT{n_bits} quantized weights', '']
    for name, arr in weights_dict.items():
        q, scale = quantize_symmetric(arr, n_bits)
        flat = q.flatten()
        shape_str = 'x'.join(str(d) for d in arr.shape)
        lines.append(f'// {name} shape=({shape_str}) count={flat.size} scale={scale:.10e}')
        lines.extend(hex_fn(int(v)) for v in flat)
        lines.append('')
        metadata['layers'][name] = {'shape': list(arr.shape), 'count': int(flat.size), 'scale': scale}
    with open(out_dir / f'model_weights_int{n_bits}.mem', 'w') as f:
        f.write('\n'.join(lines))
    with open(out_dir / 'metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)


def generate_quant_plot(results: list[dict], model_type: str, out_path: Path) -> None:
    variants = [r['variant'] for r in results]
    accuracies = [r['accuracy'] * 100 for r in results]
    times_ms = [r['inference_time_s'] * 1000 for r in results]
    sizes = [r['weight_size_bytes'] for r in results]
    colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(f'Quantization Comparison - {model_type.upper()}')
    for ax, vals, title, ylabel in zip(axes, [accuracies, times_ms, sizes], ['Accuracy', 'Inference Time', 'Weight Size'], ['Accuracy (%)', 'Time (ms)', 'Bytes']):
        bars = ax.bar(variants, vals, color=colors, edgecolor='black', linewidth=0.5)
        ax.set_title(title)
        ax.set_ylabel(ylabel)
        ymax = max(vals) if vals else 1
        for bar, val in zip(bars, vals):
            label = f'{val:.2f}' if title == 'Accuracy' else (f'{val:.1f}' if title == 'Inference Time' else f'{int(val)}')
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + ymax * 0.02, label, ha='center', va='bottom', fontsize=9)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)


def quantize_model(model_type: str, model: keras.Model | None = None, X_test: np.ndarray | None = None, y_test: np.ndarray | None = None, class_names: list[str] | None = None):
    adir = artifact_dir(model_type)
    quant_dir = adir / 'quantization'
    quant_dir.mkdir(parents=True, exist_ok=True)
    if model is None:
        model = keras.models.load_model(adir / 'har_model.keras')
    if X_test is None or y_test is None or class_names is None:
        model_cfg = TRAINING_CONFIG['models'][model_type]
        if model_cfg['data_mode'] == 'features':
            _, _, X_test_raw, y_test, _, class_names = load_har_data()
        else:
            _, _, X_test_raw, y_test, _, class_names = load_har_raw()
        if model_cfg['data_mode'] == 'wavelet':
            X_test_raw = apply_wavelet_transform(X_test_raw, wavelet=model_cfg.get('wavelet', 'db4'), level=model_cfg.get('wavelet_level', 2))
        scaler = ZScoreScaler.load(adir / 'scaler.json')
        if X_test_raw.ndim == 3:
            n, t, c = X_test_raw.shape
            X_test = scaler.transform(X_test_raw.reshape(n, t * c)).reshape(n, t, c)
            if model_type == '2dcnn':
                X_test = X_test[..., np.newaxis]
        else:
            X_test = scaler.transform(X_test_raw)
            if model_type == '1dcnn':
                X_test = X_test[..., np.newaxis]

    original = extract_weights(model)
    results = []
    for variant in ['fp32', 'fp16', 'int16', 'int8']:
        set_weights_from_dict(model, build_quantized_weights(original, variant))
        preds_probs, avg_time = timed_inference(model, X_test)
        preds = np.argmax(preds_probs, axis=1)
        per_class = {}
        for i, name in enumerate(class_names):
            mask = y_test == i
            per_class[name] = float(np.mean(preds[mask] == i)) if mask.sum() else 0.0
        results.append({
            'variant': variant,
            'accuracy': float(np.mean(preds == y_test)),
            'inference_time_s': avg_time,
            'weight_size_bytes': weight_size_bytes(original, variant),
            'per_class_accuracy': per_class,
            'total_params': int(model.count_params()),
        })
        if variant == 'fp16':
            write_mem_fp16(original, quant_dir / 'fp16')
        elif variant == 'int16':
            write_mem_int(original, 16, quant_dir / 'int16')
        elif variant == 'int8':
            write_mem_int(original, 8, quant_dir / 'int8')
    set_weights_from_dict(model, original)
    with open(quant_dir / 'results.json', 'w') as f:
        json.dump({'model_type': model_type, 'results': results}, f, indent=2)
    generate_quant_plot(results, model_type, quant_dir / 'quantization_results.png')
    return results


def load_model_result(model_type: str):
    path = artifact_dir(model_type) / 'model_spec.json'
    if not path.exists():
        return None
    with open(path, 'r') as f:
        return json.load(f)


def load_quant_result(model_type: str):
    path = artifact_dir(model_type) / 'quantization' / 'results.json'
    if not path.exists():
        return None
    with open(path, 'r') as f:
        return json.load(f).get('results', [])


def generate_comparison_plot(all_results: dict[str, dict], out_path: Path) -> None:
    models = list(all_results.keys())
    test_accs = [all_results[m]['training_info']['final_test_accuracy'] * 100 for m in models]
    params = [all_results[m]['total_params'] for m in models]
    times = [all_results[m]['training_info']['training_time_seconds'] for m in models]
    colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#AB47BC'][:len(models)]
    labels = [m.upper() for m in models]
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, vals, title, ylabel in zip(axes, [test_accs, params, times], ['Test Accuracy', 'Parameters', 'Training Time'], ['Accuracy (%)', 'Parameters', 'Time (s)']):
        bars = ax.bar(labels, vals, color=colors, edgecolor='black', linewidth=0.5)
        ax.set_title(title)
        ax.set_ylabel(ylabel)
        ymax = max(vals) if vals else 1
        for bar, val in zip(bars, vals):
            txt = f'{val:.2f}' if title == 'Test Accuracy' else (f'{int(val):,}' if title == 'Parameters' else f'{val:.1f}s')
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + ymax * 0.02, txt, ha='center', va='bottom', fontsize=9)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)


def generate_3d_plot(all_results: dict[str, dict], quant_results: dict[str, list[dict]], out_path: Path) -> None:
    xs, ys, zs, texts, colors = [], [], [], [], []
    palette = {'mlp': '#2196F3', '1dcnn': '#4CAF50', '2dcnn': '#E91E63', 'cnn_lstm': '#FF9800', 'wclstm': '#AB47BC'}
    for model_type, spec in all_results.items():
        fp32 = next((r for r in quant_results.get(model_type, []) if r['variant'] == 'fp32'), None)
        if fp32 is None:
            continue
        xs.append(spec['training_info']['final_test_accuracy'] * 100)
        ys.append(fp32['inference_time_s'] * 1000)
        zs.append(spec['total_params'])
        texts.append(model_type.upper())
        colors.append(palette.get(model_type, '#333333'))
    fig = go.Figure(data=[go.Scatter3d(x=xs, y=ys, z=zs, mode='markers+text', text=texts, textposition='top center', marker=dict(size=8, color=colors))])
    fig.update_layout(title='HAR Models: Accuracy vs Inference Time vs Parameters', scene=dict(xaxis_title='Accuracy (%)', yaxis_title='Inference Time (ms)', zaxis_title='Parameters'))
    plotly_save(fig, filename=str(out_path), auto_open=False, include_plotlyjs='cdn')


def generate_per_class_heatmap(quant_results: dict[str, list[dict]], out_path: Path) -> None:
    models = [m for m in MODEL_TYPES if m in quant_results]
    class_names = TRAINING_CONFIG['class_names']
    matrix = []
    for cls in class_names:
        row = []
        for model_type in models:
            fp32 = next((r for r in quant_results[model_type] if r['variant'] == 'fp32'), None)
            row.append((fp32['per_class_accuracy'][cls] * 100) if fp32 else 0.0)
        matrix.append(row)
    arr = np.array(matrix)
    fig, ax = plt.subplots(figsize=(max(8, len(models) * 1.6), 4.5))
    im = ax.imshow(arr, cmap='viridis', aspect='auto', vmin=0, vmax=100)
    ax.set_xticks(np.arange(len(models)), [m.upper() for m in models])
    ax.set_yticks(np.arange(len(class_names)), class_names)
    ax.set_title('Per-Class Accuracy Heatmap (FP32)')
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            ax.text(j, i, f'{arr[i, j]:.1f}', ha='center', va='center', color='white', fontsize=9)
    fig.colorbar(im, ax=ax, label='Accuracy (%)')
    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)


def generate_quantization_overview(quant_results: dict[str, list[dict]], out_path: Path) -> None:
    models = [m for m in MODEL_TYPES if m in quant_results]
    variants = ['fp32', 'fp16', 'int16', 'int8']
    x = np.arange(len(models))
    width = 0.18
    colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for idx, variant in enumerate(variants):
        acc_vals = []
        size_vals = []
        for model_type in models:
            entry = next((r for r in quant_results[model_type] if r['variant'] == variant), None)
            acc_vals.append((entry['accuracy'] * 100) if entry else 0.0)
            size_vals.append(entry['weight_size_bytes'] if entry else 0)
        axes[0].bar(x + (idx - 1.5) * width, acc_vals, width=width, label=variant.upper(), color=colors[idx])
        axes[1].bar(x + (idx - 1.5) * width, size_vals, width=width, label=variant.upper(), color=colors[idx])
    axes[0].set_xticks(x, [m.upper() for m in models])
    axes[1].set_xticks(x, [m.upper() for m in models])
    axes[0].set_title('Quantized Accuracy by Model')
    axes[1].set_title('Quantized Weight Size by Model')
    axes[0].set_ylabel('Accuracy (%)')
    axes[1].set_ylabel('Bytes')
    axes[0].legend()
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)


def write_summary_files(all_results: dict[str, dict], quant_results: dict[str, list[dict]]) -> dict:
    summary = []
    for model_type in MODEL_TYPES:
        spec = all_results.get(model_type)
        if not spec:
            continue
        fp32 = next((r for r in quant_results.get(model_type, []) if r['variant'] == 'fp32'), None)
        int8 = next((r for r in quant_results.get(model_type, []) if r['variant'] == 'int8'), None)
        summary.append({
            'model': model_type,
            'params': spec['total_params'],
            'train_accuracy': spec['training_info']['final_train_accuracy'],
            'test_accuracy': spec['training_info']['final_test_accuracy'],
            'test_loss': spec['training_info']['final_test_loss'],
            'training_time_seconds': spec['training_info']['training_time_seconds'],
            'fp32_inference_time_ms': fp32['inference_time_s'] * 1000 if fp32 else None,
            'fp32_weight_size_bytes': fp32['weight_size_bytes'] if fp32 else None,
            'int8_accuracy': int8['accuracy'] if int8 else None,
            'int8_weight_size_bytes': int8['weight_size_bytes'] if int8 else None,
        })
    payload = {'summary': summary}
    with open(ARTIFACTS_ROOT / 'summary.json', 'w') as f:
        json.dump(payload, f, indent=2)
    lines = ['model,params,train_accuracy,test_accuracy,test_loss,training_time_seconds,fp32_inference_time_ms,fp32_weight_size_bytes,int8_accuracy,int8_weight_size_bytes']
    for row in summary:
        lines.append(','.join(str(row[k]) for k in ['model', 'params', 'train_accuracy', 'test_accuracy', 'test_loss', 'training_time_seconds', 'fp32_inference_time_ms', 'fp32_weight_size_bytes', 'int8_accuracy', 'int8_weight_size_bytes']))
    with open(ARTIFACTS_ROOT / 'summary.csv', 'w') as f:
        f.write('\n'.join(lines))
    return payload


def generate_all_aggregate_outputs() -> dict:
    all_results = {}
    quant_results = {}
    for model_type in MODEL_TYPES:
        spec = load_model_result(model_type)
        if spec:
            all_results[model_type] = spec
        qr = load_quant_result(model_type)
        if qr:
            quant_results[model_type] = qr
    if all_results:
        generate_comparison_plot(all_results, ARTIFACTS_ROOT / 'comparison.png')
    if all_results and quant_results:
        generate_3d_plot(all_results, quant_results, ARTIFACTS_ROOT / 'comparison_3d.html')
        generate_per_class_heatmap(quant_results, ARTIFACTS_ROOT / 'per_class_accuracy_heatmap.png')
        generate_quantization_overview(quant_results, ARTIFACTS_ROOT / 'quantization_overview.png')
    return write_summary_files(all_results, quant_results)


def run_full_pipeline(models: list[str] | tuple[str, ...] = MODEL_TYPES, epochs: int | None = None, batch_size: int | None = None, learning_rate: float | None = None) -> dict:
    maybe_enable_gpu()
    ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)
    training_runs = {}
    for model_type in models:
        print('\n' + '=' * 80)
        print(f'Running full pipeline for {model_type.upper()}')
        print('=' * 80)
        result = train_one_model(model_type, epochs=epochs, batch_size=batch_size, learning_rate=learning_rate)
        export_model_artifacts(model_type, result['model'])
        quantize_model(model_type, model=result['model'], X_test=result['X_test'], y_test=result['y_test'], class_names=result['class_names'])
        training_runs[model_type] = {'test_accuracy': result['spec']['training_info']['final_test_accuracy'], 'params': result['spec']['total_params']}
        tf.keras.backend.clear_session()
    summary = generate_all_aggregate_outputs()
    print('\nPipeline complete. Aggregate artifacts are in:', ARTIFACTS_ROOT)
    return {'training_runs': training_runs, 'summary': summary}


In [ ]:
MODELS_TO_RUN = MODEL_TYPES
EPOCHS_OVERRIDE = None
BATCH_SIZE_OVERRIDE = None
LEARNING_RATE_OVERRIDE = None

PIPELINE_RESULTS = run_full_pipeline(
    models=MODELS_TO_RUN,
    epochs=EPOCHS_OVERRIDE,
    batch_size=BATCH_SIZE_OVERRIDE,
    learning_rate=LEARNING_RATE_OVERRIDE,
)
PIPELINE_RESULTS['summary']